# Hull checks: persistent shape or local roughening?

## Question and why it matters
Does the wider ant hull reflect persistent shape or shorter-wavelength variation?
The [registered README](README.md) fixes hypotheses before measurement. See also
[research avenue 5](../../../notes/paper-gaps-research-avenues-2026-10-15.md).
This write-up covers calibration and the bounded pilot. The full run is pending.

## Assumptions and definitions
The outermost visited-cell radius in each of 512 angular bins can miss overhangs.
The exterior is 4-connected. These are definitions, not validated physical-interface
assumptions. The unresolved census sample is not certified chaotic. Modes 3–8 are
the registered shape band, 9–64 the local band. Coarsening tests resolution.

## Provenance
Inputs below live in results/ and are intended to be committed with this experiment.
They are currently uncommitted working outputs. No raw dumps are read.


In [1]:
from pathlib import Path
import hashlib, json
SEED = 20261015
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Cargo.toml').exists())
exp = root/'experiments/archive/2026-10-hull-checks'
res = exp/'results'
manifest = json.loads((res/'pilot/manifest.json').read_text())
analysis = json.loads((res/'pilot/analysis.json').read_text())
print('producing commit:', manifest['commit'], '(dirty tree recorded in manifest)')
print('registered date: 2026-10-15 NZDT; seed:', SEED)
paths = [res/'calibration.json', res/'validation.json', res/'pilot/manifest.json', res/'pilot/analysis.json']
paths += sorted((res/'pilot').glob('*.csv'))
for p in paths:
    assert p.exists(), p
    print(p.relative_to(root), hashlib.sha256(p.read_bytes()).hexdigest())


producing commit: da88e61b690507526ba4e2a2c017d9c7941beb91 (dirty tree recorded in manifest)
registered date: 2026-10-15 NZDT; seed: 20261015
experiments/2026-10-15-hull-checks/results/calibration.json ea4a7ca36dc049863820e1af733e4b694a4eec1cff290b1ca72d900c43853e39
experiments/2026-10-15-hull-checks/results/validation.json f8de4a35cff07bc6d8472269de610ccde477b554275a36a3fc726337cc747f52
experiments/2026-10-15-hull-checks/results/pilot/manifest.json 91512baa1ff6e9099bf8a221f79815ebb7e75a39b96b98443c274408e2933c0b
experiments/2026-10-15-hull-checks/results/pilot/analysis.json d8dc3f7938c9f9fdd39b301e4dbab868b2a0f20d903f849416b5154355791647
experiments/2026-10-15-hull-checks/results/pilot/RLRLRRRLLR_331.csv d97a3b79e97bec49c8674562c781be1b008c6ebce3adaf492a540f095c37514a
experiments/2026-10-15-hull-checks/results/pilot/RLRLRRRLLR_332.csv 4665a6c2214419d4ca798d2328a921fc7961f30f2b6b870f131e59fdb844e661
experiments/2026-10-15-hull-checks/results/pilot/RLR_331.csv 8322d06e995d378c8941a63b27

## Data and method
Heavy computation is in `examples/hull_checks.rs` and `scripts/run.py`.
The pilot consists of two ant rules and one rotor, each at seeds 331 and 332,
k=16..20. It checks acquisition, exclusions and analysis. It has insufficient
rotors for the registered minimum of eight radius matches and cannot test H-shape/H-local.

## Results


In [2]:
for name in ('calibration.json', 'validation.json'):
    d = json.loads((res/name).read_text())
    print(name, d['status'])
print('Valid profiles:', len(analysis['profiles']))
print('Excluded profiles/escape records:', len(analysis['exclusions']))
print('Missing files:', analysis['missing_files'])
print('Exclusion reasons:')
for s in analysis['exclusions']: print(' ', s)
print('Decisions:', {k:v['decision'] for k,v in analysis['verdicts'].items()})


calibration.json PASS
validation.json PASS
Valid profiles: 10
Excluded profiles/escape records: 20
Missing files: []
Exclusion reasons:
  RLRLRRRLLR@331 k=16: 7 empty bins
  RLRLRRRLLR@331 k=17: 7 empty bins
  RLRLRRRLLR@331 k=18: 2 empty bins
  RLRLRRRLLR@332 k=16: 14 empty bins
  RLRLRRRLLR@332 k=17: 4 empty bins
  RLRLRRRLLR@332 k=18: 6 empty bins
  RLR@331 k=16: 10 empty bins
  RLR@331 k=17: 5 empty bins
  RLR@331 k=18: 4 empty bins
  RLR@331 k=19: 2 empty bins
  RLR@331 k=20: 1 empty bins
  RLR@332 k=16: 19 empty bins
  RLR@332 k=17: 6 empty bins
  RLR@332 k=18: 2 empty bins
  RLR@332 k=20: 1 empty bins
  rotor:331 k=16: 7 empty bins
  rotor:331 k=17: 1 empty bins
  rotor:331 k=18: 1 empty bins
  rotor:332 k=16: 3 empty bins
  rotor:332 k=17: 2 empty bins
Decisions: {'legacy': 'pilot only', 'sampled': 'pilot only'}


The estimator calibration passes and the pilot retains populated profiles while
listing sparse-bin failures explicitly. This validates plumbing; it does not
show whether the ants' residual hull fluctuations exceed matched rotors.

## Caveats and controls
The small pilot has sparse angular bins, so invalid profiles must not be silently
interpolated. Synthetic controls check mode assignment, Parseval including Nyquist,
the exact sinusoid structure function and persistence of lobes versus orthogonal
local modes. Compatibility validation checks nine original-instrument rows and
byte equality with one versus six Rayon threads. The resume log records six skips.
The full run uses absolute power, matched radii and bin coarsening, addressing the
normalisation problem in the earlier hull analysis. No scientific prediction has
been tested by this pilot.

## What this changes / next steps
Run the registered full cohort and evaluate the matching/resolution gates before
interpreting either hypothesis. The pilot demonstrates a reproducible instrument.
No new empirical claim about asymptotic roughening is warranted.

## Open questions
Do low modes persist? Is there residual local excess at equal radii? Does the
estimated correlation length have adequate resolution and a reproducible scaling?
The unresolved-rule sample may contain transient builders or highways.
